# 🚀 EchoVision: Decoupled Audio-Visual RAG for Video QA
### ⚡ Google Colab T4 GPU Edition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**EchoVision** is an advanced multi-modal Retrieval-Augmented Generation (RAG) framework designed for Video Question Answering (Video QA). It decouples audio and visual streams to eliminate visual dominance and avoid hallucination:
- 🔊 **Audio Stream**: `Whisper-large-v3` (speech) + `LAION-CLAP` (acoustic sound events) $\rightarrow$ **ChromaDB**.
- 👁️ **Visual Stream**: `PySceneDetect` + 4-metric Image Quality Assessment + `Qwen2.5-VL-3B` frame descriptions + `CLIP` embeddings $\rightarrow$ **FAISS**.
- 🧠 **Cross-Modal Reasoning**: Dynamic modality gate $\beta(q)$, decoupled parallel retrieval, audio-aware cross-encoder re-ranking (`bge-reranker-large`), temporal NMS, and grounded answer generation.

---
### 🛠️ Hardware Requirements:
- **GPU**: NVIDIA Tesla T4 (15-16 GB VRAM) — *Standard Google Colab Free Tier*
- **Colab Setting**: Make sure GPU acceleration is enabled: **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ **T4 GPU**.


## 1️⃣ Hardware Verification: Confirm Tesla T4 GPU
Run this cell to verify your GPU type, VRAM, and PyTorch CUDA configuration.


In [ ]:
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM      : {vram_gb:.2f} GB")
    print(f"CUDA Capability : {torch.cuda.get_device_capability(0)}")
    print("[✓] Optimal runtime: Tesla T4 detected!")
else:
    print("[!] GPU not detected. Please select 'Runtime' -> 'Change runtime type' -> 'T4 GPU'.")
print("=" * 60)


## 2️⃣ Clone Repository or Setup Project Files
Choose whichever method fits your workflow:
- **Option A**: Clone directly from GitHub.
- **Option B**: Mount your Google Drive (if project files are in Drive).
- **Option C**: Use the current folder if you already uploaded this repository.


In [ ]:
import os

# OPTION A: Clone repository from GitHub (uncomment and replace URL):
# !git clone https://github.com/YOUR_USERNAME/cloud_EVL.git
# %cd cloud_EVL

# OPTION B: Mount Google Drive (uncomment if using Google Drive):
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/cloud_EVL

print("Current Working Directory:", os.getcwd())
!ls -lh


## 3️⃣ Install System & Python Dependencies
Install `ffmpeg`, `libsndfile1`, and all required multi-modal Python packages.


In [ ]:
# 1. System packages for audio & video extraction
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1

# 2. Multi-modal Python packages
!pip install -q -r requirements.txt

# Verify key imports
import faster_whisper
import transformers
import scenedetect
import chromadb
import faiss
print("\n[✓] All dependencies installed successfully!")


## 4️⃣ Stage 3 LLM Generation Setup (Dual-Backend)
EchoVision features a **dual-backend generator**:
- **Mode 1: Ollama Server (Recommended)**: Runs `ollama serve` in the background with `qwen2.5:1.5b` (or `qwen2.5:7b`). Extremely fast and only takes ~1.5GB VRAM on T4!
- **Mode 2: Hugging Face In-Process**: Runs `Qwen/Qwen2.5-1.5B-Instruct` natively in PyTorch with zero background servers.

Run the cell below to install Ollama and pull the model:


In [ ]:
# Install Ollama binary in Colab
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server as a background daemon
import subprocess, time, requests
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

# Pull lightweight, high-performance Qwen2.5 model (ideal for T4 GPU)
!ollama pull qwen2.5:1.5b

# Verify Ollama server status
try:
    res = requests.get("http://localhost:11434/api/tags", timeout=2.0)
    models = [m['name'] for m in res.json().get('models', [])]
    print(f"[✓] Ollama active! Available models: {models}")
except Exception as e:
    print("[!] Ollama not running; the pipeline will automatically use the in-process Hugging Face LLM backend.")


## 5️⃣ Hardware & Model Configuration for T4 GPU
We set environment variables tailored for the Tesla T4 (16GB VRAM):
- `WHISPER_MODEL`: `large-v3` (SOTA speech transcription in float16)
- `OLLAMA_MODEL`: `qwen2.5:1.5b` (low latency, high reasoning quality)
- `LLM_BACKEND`: `auto` (detects Ollama automatically, falls back to HF)
- `CONCURRENT_INGESTION`: `0` (**Sequential extraction**: extracts audio then visual, keeping peak VRAM under 7.5 GB on T4!)


In [ ]:
import os

os.environ["WHISPER_MODEL"] = "large-v3"
os.environ["OLLAMA_MODEL"] = "qwen2.5:1.5b"
os.environ["HF_LLM_MODEL"] = "Qwen/Qwen2.5-1.5B-Instruct"
os.environ["LLM_BACKEND"] = "auto"
os.environ["CONCURRENT_INGESTION"] = "0"  # Memory-safe sequential mode for T4

import config
print("=" * 50)
print(f"Configured Device      : {config.DEVICE}")
print(f"Configured Precision   : {config.TORCH_DTYPE}")
print(f"Configured LLM Backend : {config.LLM_BACKEND}")
print(f"Ingestion Mode         : {'Sequential (Memory-Safe for T4)' if not config.CONCURRENT_INGESTION else 'Concurrent'}")
print("=" * 50)


## 6️⃣ Inspect Videos & Question Dataset
Discover videos in `smoketest/videos` and verify questions schema in `smoketest/json`:


In [ ]:
from ingestion import discover_videos
import json

videos = discover_videos("smoketest/videos")
print(f"Discovered {len(videos)} video(s) in smoketest/videos:")
for v in videos[:8]:
    print(f"  - {os.path.basename(v)}")
if len(videos) > 8:
    print(f"  ... and {len(videos) - 8} more")

json_file = "smoketest/json/smoketest_questions.json"
if os.path.exists(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        q_data = json.load(f)
    print(f"\nTotal Questions in Dataset: {len(q_data)}")
    print("Sample Question:")
    print(json.dumps(q_data[0], indent=2))


## 7️⃣ Run Stage 1 (Offline Feature Extraction & Isolated Vector Indexing)
- Scans `smoketest/videos/`
- Extracts Speech (`Whisper-large-v3`) & Acoustic Sound Events (`CLAP`) $\rightarrow$ **ChromaDB**
- Extracts Keyframes (`PySceneDetect` + Image Quality Filter + CLIP deduplication) + VLM scene descriptions (`Qwen2.5-VL-3B`) $\rightarrow$ **FAISS**
- Saves isolated vector stores per video in `data/vector_stores/<video_stem>_<hash>/`
- Automatically skips already-indexed videos!


In [ ]:
# Run Stage 1 Ingestion with sequential memory safety on T4
!python ingestion.py --dataset_dir smoketest --sequential


## 8️⃣ Run Stage 2 & 3 (Decoupled Retrieval & Answer Generation)
- Discovers pre-built vector stores from Stage 1
- Dynamically estimates audio/visual modality dependency $\beta(q)$
- Decoupled parallel retrieval from ChromaDB and FAISS
- Audio-aware cross-encoder re-ranking (`bge-reranker-large` in fp16)
- Temporal NMS deduplication & sufficiency verification
- Generates concise grounded answers and outputs to `output/smoketest_questions.json`!


In [ ]:
!python main.py --dataset_dir smoketest --output_dir output


## 9️⃣ View Predictions & Evaluation Metrics Summary
Inspect exact match and relaxed accuracy metrics across spatial, temporal, spatiotemporal, and audio categories:


In [ ]:
import json
import pandas as pd

eval_path = "output/evaluation_summary.json"
if os.path.exists(eval_path):
    with open(eval_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    print("=" * 55)
    print("               EVALUATION METRICS REPORT               ")
    print("=" * 55)
    metrics = eval_data.get("metrics", {})
    print(f"Total Evaluated Questions   : {metrics.get('total_evaluated', 0)}")
    print(f"Exact Match Accuracy        : {metrics.get('exact_match_accuracy', 0)}%")
    print(f"Relaxed Substring Accuracy  : {metrics.get('relaxed_accuracy', 0)}%")
    if metrics.get("category_breakdown"):
        print("\nCategory Breakdown:")
        for cat, c_res in metrics["category_breakdown"].items():
            print(f"  - {cat:<16}: {c_res['accuracy']:>6.2f}% ({c_res['correct']}/{c_res['total']})")
    print("=" * 55)

pred_path = "output/smoketest_questions.json"
if os.path.exists(pred_path):
    with open(pred_path, "r", encoding="utf-8") as f:
        preds = json.load(f)
    df = pd.DataFrame(preds)
    cols = [c for c in ["video_id", "category", "question", "predicted_answer", "ground_truth_answer"] if c in df.columns]
    print(f"\nSample Predictions Table ({len(df)} total):")
    display(df[cols].head(15))


## 🔟 Run Scientific Ablation Benchmark Suite (`run_ablations.py`)
Run all 4 ablations + baseline in one automated sweep to empirically validate the architecture:
1. `joint_store` (Unified vs Decoupled vector stores)
2. `no_audio_lift` (Re-ranking without audio lift)
3. `fixed_cutoff` (Fixed depth vs Adaptive sufficiency gate)
4. `no_audio` (Audio removed entirely - visual-only baseline)


In [ ]:
# Run full ablation benchmark over the dataset
!python run_ablations.py --dataset_dir smoketest --output_dir output/ablations


## 💡 Interactive Demo: Query Any Video with Custom Questions
Test any video file interactively! Specify a video path and question below:


In [ ]:
import os
from ingestion import discover_videos, Stage1Ingestor
from main import answer_question_for_video, AblationConfig
from stage2_online.question_classifier import QuestionClassifier
from stage2_online.deduplicator import Deduplicator
from stage2_online.reranker import ReRanker
from stage2_online.sufficiency_gate import SufficiencyGate
from stage3_generator.generator import Generator

sample_videos = discover_videos("smoketest/videos")
if sample_videos:
    test_video = sample_videos[0]  # Or specify any path like "smoketest/videos/v_0q9yZPTBbus.mp4"
    test_question = "what is behind the person in blue?"
    
    print(f"Selected Video: {test_video}")
    print(f"Question      : {test_question}\n")
    
    # 1. Run Stage 1 (instant if already indexed)
    ingestor = Stage1Ingestor()
    indexer, reused, err = ingestor.process_single_video(test_video)
    
    if indexer:
        # 2. Shared online reasoning components
        shared_components = {
            'qc': QuestionClassifier(),
            'dedup': Deduplicator(),
            'reranker': ReRanker(),
            'gate': SufficiencyGate(),
            'generator': Generator()
        }
        
        # 3. Answer question
        answer = answer_question_for_video(indexer, test_question, shared_components)
        print("\n" + "=" * 50)
        print(f"PREDICTED ANSWER: {answer}")
        print("=" * 50)


## 💾 Export & Download Output Files
Download the evaluation summary, predictions JSON, and ablation reports directly to your computer.


In [ ]:
from google.colab import files
import shutil

shutil.make_archive("echovision_colab_results", 'zip', "output")
files.download("echovision_colab_results.zip")
        print("[OK] Results archive downloaded!")
